# Dialog servers and browser ES modules
> Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.

In [ ]:
#| default_exp server

In [ ]:
#| export
#| hide
import asyncio, inspect, json, keyword, logging, os, subprocess
from contextlib import asynccontextmanager
from collections.abc import Callable, AsyncIterator
from datetime import datetime
from functools import partial
from urllib.parse import quote
from starlette.responses import Response
from starlette.routing import Route
from fastcore.basics import first
from fastcore.aio import maybe_await
from fastcore.meta import delegates
from fastcore.net import is_port_free
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, JupyUviAsync, HTMX
from fasthtml.xtend import Script, Link
from dialoghelper.core import js_eval_a, read_msg
from pote.basic import Kounter
from pote.dutil import link_msg, set_var

In [ ]:
#| hide
import socket
from multiprocessing import Queue
import fastcore.docments
from itertools import count
from unittest.mock import patch as mock_patch
from starlette.testclient import TestClient
from ipykernel_helper import info_md
from fastcore.basics import setify, ifnone
from fastcore.foundation import store_attr, patch
from fastcore.parallel import startproc
from fastcore.test import *
from fasthtml.common import fast_app
from pyskills import doc
from pyskills import *
from fasthtml.components import Div, Span, P
from dialoghelper.core import add_msg, iife_a, add_html_a, add_scr_a, js_run_a, del_msgs
from htmx_bridge import get_hx
from pote.logger import logger
from pote.dutil import wait_for_a

In [ ]:
#| exporti
kntr = Kounter()

In [ ]:
hx = get_hx()

In [ ]:
try:
    if globals()['srv']: await maybe_await(srv.stop())
except BaseException: pass
srv=app=rt=p = None

### Choosing a helper

```python
from pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg
```

- `await ensure_server_a(...)` starts or reuses a persistent FastHTML server and returns `(srv, app, rt, p)`.
- `async with serve_app(...)` is for a scoped server that is stopped automatically, particularly in tests.
- `find_server`, `is_serving_a`, and `stop_server_a` inspect and clean up existing server state.
- `free_port(public=True)` returns first available public port.
- `p(path)` previews a route in an iframe.
- `setup_mod(..., versioned=True)` supports iterative browser ES-module work.
- `get_js`, `add_js_msg`, and `setup_mod` without `src` extract JavaScript from a `%%js`/`%%javascript` message.

The async APIs are normally preferable in Solveit and Jupyter. These helpers target FastHTML applications; the underlying `JupyUvi` can serve more general ASGI applications, but `pote.server` does not promise that compatibility.

## references

In [ ]:
# from inspect import getsource
# import fasthtml.jupyter
# await add_msg(getsource(fasthtml.jupyter))

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

Does JupyUvi need a FastHTML app or any Starlette app will do?

---

`JupyUvi` only needs an **ASGI app**, so a plain Starlette app will do.

The constructor passes `app` straight into:

```python
uvicorn.Config(app, log_level=log_level, host=host, port=port, **kwargs)
```

so uvicorn just needs something ASGI-callable.

The only FastHTML-specific-ish part is optional `live=True`:

```python
if live: self._setup_live(app)
```

`_setup_live` assumes the app has FastHTML-style conveniences:

```python
app.hdrs.append(...)
@app.get(rt)
async def _sse(): ...
```

So:

- `JupyUvi(app)` works with any ASGI app Starlette/ FastAPI/FastHTML, etc.
- `JupyUvi(app, live=True)` expects an app with `.hdrs` and `.get`, so that's really for FastHTML-style apps unless your Starlette app provides compatible attributes.
- `HTMX(..., path=FT_or_tuple, app=app)` also assumes `app.get(route)(...)`, so that convenience path is FastHTML/Starlette-route-decorator compatible.

For a normal Starlette app:

```python
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.routing import Route

async def home(request): return PlainTextResponse('hi')

app = Starlette(routes=[Route('/', home)])
srv = JupyUvi(app, port=8000)
```

That should work fine.

In [ ]:
doc(fastcore.docments)

# module fastcore.docments:

"""Document parameters using comments.

A *docment* documents a parameter with a trailing comment on its line in the signature, right next to the name, type, and default it describes; `docments(f)` retrieves them programmatically, so tools (docs tables, CLIs like `fastcore.script`, ...) can use them. This replaces repeating parameter names in a specially-formatted docstring, which has to be kept in sync with the signature by hand:

```python
def add(
    a:int,  # The first operand
    b=1,    # The second operand
) -> int:   # The sum
    "Add `a` to `b`"
    return a + b
test_eq(docments(add), {'a': 'The first operand', 'b': 'The second operand', 'return': 'The sum'})
```

Comment lines *above* a param also work when one line isn't enough; `typing.Annotated` string metadata is an alternative to comments; numpy-style docstring sections are merged in when present; and `docments(f, full=True)` returns each param's default, type, and docment together.

Docs: 

## helpers

In [ ]:
#| exporti
def kill_port(port):
    "Forcibly terminate local processes listening on `port`; intended only for notebook recovery"
    subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
def listen(q):
    s = socket.socket(); s.bind(('', 0)); s.listen()
    q.put(s.getsockname()[1]); s.accept()

q = Queue()
proc = startproc(lambda: listen(q), daemon=True)
port = q.get(timeout=5)
kill_port(port)
proc.join(5)

test_eq(proc.is_alive(), False)
test_is(is_port_free(port), True)

What are we testing here really, Python's `subprocess`; unix shell's `lsof`?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It is an **integration test for `kill_port`**, not a unit test of either `subprocess` or `lsof`.

It verifies this end-to-end contract:

1. A separate process opens and listens on a dynamically allocated port.
2. `kill_port(port)` runs its `lsof | xargs kill -9` command.
3. The listening process terminates.
4. The port becomes bindable again.

These assertions cover the intended observable effects:

```python
test_eq(proc.is_alive(), False)
test_is(is_port_free(port), True)
```

So it necessarily depends on Unix tools (`lsof`, `xargs`, `kill`) and `subprocess` working, but it does not independently test their implementations. It tests that the particular recovery helper works in the environment where it is meant to run.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 38164, "completion_tokens": 299, "total_tokens": 38463, "cached_tokens": 3328, "reasoning_tokens": 124, "cost": 0.00594885}
```


And that woud be very useful for a external library deciding to use `pote.server.killport`. But in this notebook, right after writing a function that in one line use Python's `subprocess` to call 'lsof`, what's the point of testing it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Very little, beyond catching a typo or portability failure in that shell command.

`kill_port` contains no meaningful Python logic: it delegates entirely to `subprocess` and the Unix pipeline. The test confirms the exact command works *on this machine now*, but does not establish a library-level behavior independently of those external tools.

In this notebook, I would treat it as a one-off manual smoke check while writing the helper, not a durable test:

```python
kill_port(port)
test_is(is_port_free(port), True)
```

Even that is arguably redundant if `kill_port` is explicitly a notebook-recovery convenience rather than part of the supported public API. A package test only becomes worthwhile if the helper’s cross-platform/environment contract is deliberate and supported.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 38448, "completion_tokens": 263, "total_tokens": 38711, "cached_tokens": 37120, "reasoning_tokens": 104, "cost": 0.0059644500000000005}
```


The only usefuf bit of the testing is that I don't want to use a literal port so this notebook runs fine in most cases.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly. The useful part is the **setup**, not the assertions:

```python
q = Queue()
proc = startproc(lambda: listen(q), daemon=True)
port = q.get(timeout=5)
```

That obtains an actually occupied, unpredictable port, avoiding a collision with an existing service or another notebook run.

After that, the important cleanup action is simply:

```python
kill_port(port)
proc.join(5)
```

The `test_eq(proc.is_alive(), False)` and `test_is(is_port_free(port), True)` are optional confirmation. The dynamic-port setup makes the notebook reliably runnable; the test merely verifies the cleanup did what it was asked to do.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 38690, "completion_tokens": 141, "total_tokens": 38831, "cached_tokens": 8448, "cost": 0.0059092500000000004}
```


In [ ]:
#| export
def wrap_endp(
    response, # Existing request handler or fixed response value
) -> Callable:
    "Return callable handlers unchanged; otherwise return an async handler that returns `response` unchanged."
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
fixed = 'ok'
ep = wrap_endp(fixed)
test_eq(await ep(None), fixed)

ft = Div('x')
test_is(await wrap_endp(ft)(None), ft)

seq = (Div('a'), Span('b'))
test_is(await wrap_endp(seq)(None), seq)

sync_handler = lambda request: 'sync'
test_is(wrap_endp(sync_handler), sync_handler)

async def async_handler(request): return 'async'
test_is(wrap_endp(async_handler), async_handler)

component = lambda: None
component.__ft__ = lambda: Div('x')
test_is(await wrap_endp(component)(None), component)

In [ ]:
json.loads(os.environ["PUBLIC_DOMAINS"]).keys()

dict_keys(['6000', '8000', '8001', '8002', '8003', '8004', '8005', '8006', '8007'])

In [ ]:
#| export
def pub_ports() -> tuple[int]:
    "Public ports reserved by this Solveit instance."
    return tuple(sorted(map(int, json.loads(os.environ.get('PUBLIC_DOMAINS', '{}')))))

In [ ]:
#| export
def free_port(
    start:int=6550, # First port to consider
    skip=(), # Ports to exclude even when available
    public:bool=False, # Restrict search to this instance's public ports?
    host:str='127.0.0.1', # Host interface used to test availability
) -> int:
    "Return the first available port; raise `RuntimeError` when none are available."
    skip, ports = set(skip), pub_ports() if public else range(start, 65536)
    if (port := first(p for p in ports if p not in skip and is_port_free(p, host))) is None: raise RuntimeError("No free port")
    return port

In [ ]:
start = free_port()
next_free = free_port(start=start+1)

test_eq(free_port(start=start), start)
test_eq(free_port(start=start, skip=[start]), next_free)
test_is(free_port(public=True) in pub_ports(), True)

with mock_patch.dict(free_port.__globals__, is_port_free=lambda *args: False):
    with expect_fail(RuntimeError, 'No free port'): free_port(public=True)

free_port(public=True)

6000

In [ ]:
#| export
def free_public_domain():
    "Public domain for the first public port that is currently free."
    return json.loads(os.environ["PUBLIC_DOMAINS"])[str(free_port(public=True))]

In [ ]:
with mock_patch.dict(free_public_domain.__globals__, free_port=lambda public: 8000):
    test_eq(free_public_domain(), json.loads(os.environ["PUBLIC_DOMAINS"])['8000'])

In [ ]:
#| export
def find_server(
    port:int|None=None, # Port to match; `None` matches any port
    all_ports:bool=False, # Return every started binding across all ports; aliases may repeat the same server
) -> tuple[JupyUvi,str]|tuple[None,None]|list[tuple[JupyUvi,str]]:
    "Find namespace-bound `JupyUvi` objects whose flags indicate they are started; no HTTP health check. First-match order follows the IPython namespace."
    import IPython
    if all_ports: port = None
    ns = IPython.get_ipython().user_ns
    res = [(v,k) for k,v in ns.items() if isinstance(v, JupyUvi) and (port is None or v.port == port) and v.server and v.server.started and not v.server.should_exit]
    return res if all_ports else res[0] if res else (None,None)

In [ ]:
srv = JupyUvi(app := FastHTML(), port=free_port())

In [ ]:
srv2,_ = find_server(srv.port)
test_is(srv2, srv)

In [ ]:
test_is(find_server(srv.port)[0], srv)
test_eq(find_server()[0], srv2)
test_eq(find_server(None, all_ports=True), find_server(all_ports=True))
test_eq(set([_[1] for _ in find_server(all_ports=True)]), {'srv', 'srv2'})

In [ ]:
srv.stop()
await asyncio.sleep(0.2)
test_eq(find_server(None, all_ports=True), [])

In [ ]:
test_eq(find_server(), (None,None))
test_eq(find_server(None, all_ports=True), [])

In [ ]:
#| export
def _conn_host(host): return '127.0.0.1' if host in ('0.0.0.0', '::') else host

def is_serving(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        return httpx.get(f"http://{_conn_host(srv.host)}:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

async def is_serving_a(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        async with httpx.AsyncClient(timeout=timeout) as cli: return (await cli.get(f"http://{_conn_host(srv.host)}:{srv.port}/")).status_code < 500
    except Exception: return False

In [ ]:
test_eq(is_serving(None), False)
test_eq(await is_serving_a(None), False)

In [ ]:
p = free_port(6551, skip=pub_ports())
srv = JupyUvi(FastHTML(), port=p)
test_eq(is_serving(srv), True)
test_eq(await is_serving_a(srv), True)
srv.stop()
await asyncio.sleep(0.2)
test_eq(is_serving(srv), False)
test_eq(await is_serving_a(srv), False)

# HTTP-500 health
p = free_port(6551, skip=pub_ports())
app = FastHTML()
@app.get('/')
def unhealthy(): return Response(status_code=500)
srv = JupyUvi(app, port=p)
test_eq((is_serving(srv), await is_serving_a(srv)), (False,False))
srv.stop()
await asyncio.sleep(0.2)

In [ ]:
#| exporti
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
def mk_app(app_kw=None):
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return FastHTML(**app_kw)

In [ ]:
a = mk_app({})
test_eq(all(h not in a.hdrs for h in daisyui_hdrs), True)

## server

In [ ]:
#| export
def stop_server(port:int|None=None) -> None:
    "Stop the discovered synchronous server and clear every namespace alias to it; do nothing if unmatched and raise for `JupyUviAsync`."
    srv,_ = find_server(port)
    if not srv: return
    if inspect.iscoroutinefunction(srv.stop): raise RuntimeError("JupyUviAsync requires `await stop_server_a(port)`")
    if is_serving(srv): srv.stop()
    for k,v in list(get_ipython().user_ns.items()):
        if v is srv: set_var(k, None)

async def stop_server_a(port:int|None=None) -> None:
    "Stop a discovered synchronous or asynchronous server and clear every namespace alias to it; do nothing if unmatched."
    srv,_ = find_server(port)
    if not srv: return
    if await is_serving_a(srv): await maybe_await(srv.stop())
    for k,v in list(get_ipython().user_ns.items()):
        if v is srv: set_var(k, None)

In [ ]:
p = free_port(6551, skip=pub_ports())
tst_srv = JupyUvi(mk_app({}), port=p)
test_is(find_server(p)[0], tst_srv)
test_eq(is_serving(tst_srv), True)
stop_server(p)
await asyncio.sleep(0.2)
test_eq(find_server(p), (None,None))
test_eq(tst_srv, None)

In [ ]:
tst_srv_a = JupyUviAsync(mk_app({}), port=p)
await tst_srv_a.start()
test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)

with expect_fail(RuntimeError, 'stop_server_a'): stop_server(p)

test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)

await stop_server_a(p)
test_eq(find_server(p), (None,None))
test_eq(tst_srv_a, None)

In [ ]:
#| exporti
def _h(port): return '0.0.0.0' if port in pub_ports() else '127.0.0.1'

def get_preview(app, port=8000, host=None):
    "Return a function that displays the app in an iframe."
    if host is None: host = None if port==8000 else json.loads(os.environ['PUBLIC_DOMAINS']).get(f'{port}', '127.0.0.1')
    return partial(HTMX, app=app, host=host, port=port if port in (8000, None) else 443)

def srv_bundle(srv):
    if hasattr(srv.app, 'route'): return srv,srv.app,srv.app.route,get_preview(srv.app, srv.port) if srv.port in pub_ports() else None
    else: return srv,srv.app,None,None

def _server_target(srv=None, port:int|None=None, reuse=True, host='127.0.0.1'):
    own = False
    if srv: return srv,port,own
    if port is None: port = free_port(6551, skip=pub_ports(), host=host)
    found,_ = find_server(port)
    if found:
        if not reuse: raise RuntimeError(f"Server already running on port {port}")
        return found,port,own
    if not is_port_free(port): raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
    return None,port,True

def _ensure_server(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, **jupyuvi_kw):
    srv,port,own = _server_target(srv, port, reuse, jupyuvi_kw.get('host', '127.0.0.1'))
    if srv:
        if not is_serving(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv, own
    srv = JupyUvi(app if app is not None else mk_app(app_kw), port=port, **jupyuvi_kw)
    return srv, own

async def _ensure_server_a(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, **jupyuvi_kw):
    srv,port,own = _server_target(srv, port, reuse, jupyuvi_kw.get('host', '127.0.0.1'))
    if srv:
        if not await is_serving_a(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv, own
    srv = JupyUviAsync(app if app is not None else mk_app(app_kw), port=port, **jupyuvi_kw)
    await srv.start()
    return srv, own

In [ ]:
srv, own = _ensure_server()
test_is(srv is not None, True)
test_is(own, True)

srv2, _ = await _ensure_server_a(port=srv.port)
test_is(srv, srv2)

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
await stop_server_a()

test_is(globals()['srv'], None)
test_is(globals()['srv2'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

Test explicit-port start/reuse/stop.

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,_ = _ensure_server(port=port)
test_eq((srv.port, is_serving(srv)), (port, True))

srv2,_ = _ensure_server(port=port)
test_is(srv2, srv)

await stop_server_a(port)


`port=None` chooses a non-public free port and starts cleanly:

In [ ]:
srv,_ = _ensure_server()
test_eq(srv.port in pub_ports(), False)
test_eq(srv.port >= 6551, True)

await stop_server_a(srv.port)

Passing the same app is accepted and reused:

In [ ]:
port = free_port(skip=pub_ports())
app = FastHTML()
srv1,_ = _ensure_server(port=port, app=app)
srv2,_ = _ensure_server(port=port, app=app)
test_is(srv1.app, srv2.app)

await stop_server_a(port)

In [ ]:
dead,_ = _ensure_server(port=None)
dead_ref = [dead]

dead.stop()
await asyncio.sleep(0.2)

test_eq(is_serving(dead_ref[0]), False)
test_fail(lambda: _ensure_server(srv=dead_ref[0]), contains='not serving', exc=RuntimeError)

dead_ref.clear()

Test the “existing server wins” contract explicitly, including that a passed `app` is ignored when a server already exists on the requested port.

In [ ]:
port = free_port(skip=pub_ports())
app1,app2 = FastHTML(),FastHTML()
srv,_ = _ensure_server(port=port, app=app1)
srv2,_ = await _ensure_server_a(port=port, app=app2)

test_is(srv2, srv)
test_is(srv2.app, app1)

await stop_server_a(port)

In [ ]:
srv,own = _ensure_server()
srv2,own2 = await _ensure_server_a(port=srv.port)

test_eq((own, own2), (True, False))
await stop_server_a(srv.port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)

with expect_fail(RuntimeError, 'already running'):
    await _ensure_server_a(port=port, reuse=False)
test_eq(await is_serving_a(srv), True)

await stop_server_a(port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)
ref = [srv]
await stop_server_a(port)

with expect_fail(RuntimeError, 'not serving'):
    await _ensure_server_a(srv=ref[0])

test_eq(await is_serving_a(ref[0]), False)
ref.clear()

In [ ]:
def ensure_server(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', **jupyuvi_kw):
    "Return a live server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, **jupyuvi_kw)
    return (*srv_bundle(srv),)

async def ensure_server_a(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', **jupyuvi_kw):
    "Return a live async server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, **jupyuvi_kw)
    return (*srv_bundle(srv),)

In [ ]:
mid1 = (await read_msg(0)).id

port = free_port(skip=(6000,), public=True)
domain = json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]

srv, app, rt, p = await ensure_server_a(port=port, host='0.0.0.0')
test_eq((srv.port, await is_serving_a(srv)), (port, True))

@rt
def hello(): return f"Running on port {port}"

p(hello)

In [ ]:
mid2 = (await read_msg(0)).id

url = '/hello' if port == 8000 else f"https://{domain}/hello"
(c := P(hx_get=url, hx_trigger='load once', id='hello01'))

In [ ]:
await wait_for_a(lambda: hx.eval(f'''$('#{c}').text()''')==f"Running on port {port}")

In [ ]:
await update_msg(mid1, output='')
await update_msg(mid2, output='')
await stop_server_a()

In [ ]:
try:
    port = free_port(6551, skip=pub_ports())
    srv,app,rt,p = ensure_server(port=port)
    test_is(p, None)
finally:
    await stop_server_a(port)

`ensure_server` returns the standard notebook/Solveit serving bundle: a live server, the FastHTML app it serves, the app route decorator, and a preview helper wired to that server.

Existing servers have priority. If a server is passed explicitly, it must be live and is used. Otherwise, an already-running server on the requested port is reused. If no matching server exists, a new one is started, using the provided app if any, otherwise creating a default app.

The returned preview helper always targets the returned server's port.

For the synchronous server returned here, `stop_server` stops the server and clears its notebook namespace aliases; use `await stop_server_a(...)` for asynchronous or general cleanup.

## esm loader

In [ ]:
__mods__ = {}
def reg_mod(
    nm:str, # Kernel-global registry key; versioned entries use `<nm>_<version>`
    src:str, # JavaScript module source served for this key
) -> None:
    "Register or replace JavaScript source; entries persist in this kernel across app and server lifetimes."
    __mods__[nm] = src
def _get_mod(
    nm:str, # Base module name from the request route
    v:int=0, # Requested version; resolves `<nm>_<v>` before the base name
    debugger:bool=False, # Prefix existing module source with `debugger;`
):
    "Return registered JavaScript as an HTTP response, or a diagnostic module when missing."
    src = __mods__.get(f'{nm}_{v}', __mods__.get(nm))
    if src is None: src,debugger = f'console.log("[{nm}] ES module version `{v}` not found");',False
    return Response(('debugger;\n' if debugger else '')+src, media_type='text/javascript')

In [ ]:
#| exporti
def _bool_q(o): return str(o).lower() in ('1','true','yes','on')

async def esm_endpoint(req):
    pp = req.path_params
    return _get_mod(pp['nm'], int(req.query_params.get('v', 0)), _bool_q(req.query_params.get('debugger', False)))

In [ ]:
#| exporti
esm_routes = (Route('/esm/{nm}', esm_endpoint), )

In [ ]:
def setup_loader(app=None, mods=None):
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs, routes=esm_routes)
    else: app_kw = {**app_kw, 'routes': tuple(app_kw.get('routes', ()))+esm_routes}
    return FastHTML(**app_kw)

In [ ]:
a = mk_app({})
test_eq(any(r.path == '/esm/{nm}' for r in a.routes), True)
# test_eq(any(r.path == '/esm/{nm}?v={v}' for r in a.routes), True)

In [ ]:
__mods__.clear()
a = mk_app({})
setup_loader(mods=dict(a='console.log("a")', a_2='console.log("a", 2)'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=1').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=2').text, 'console.log("a", 2)')

In [ ]:
__mods__.clear()
a = mk_app({})
cli = TestClient(a)

test_eq(cli.get('/esm/nope').headers['content-type'], 'text/javascript; charset=utf-8')
test_eq(cli.get('/esm/nope').text, 'console.log("[nope] ES module version `0` not found");')
test_eq(cli.get('/esm/nope?v=3&debugger=true').text, 'console.log("[nope] ES module version `3` not found");')

In [ ]:
#| exporti
def add_esm_routes(app):
    paths = {r.path for r in app.routes}
    for r in esm_routes:
        if r.path not in paths: app.routes.append(r)
    return app

In [ ]:
def setup_loader(
    app:FastHTML|None=None, # App receiving the ESM routes once; omit to register source without modifying an app
    mods:dict[str,str]|None=None, # Module names and sources added to the kernel-global registry
) -> None:
    "Idempotently install `/esm/{nm}` on `app` and register or replace kernel-global module sources."
    if app is not None: add_esm_routes(app)
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

In [ ]:
#| exporti
def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return add_esm_routes(FastHTML(**(app_kw or {})))

In [ ]:
__mods__.clear()
a = FastHTML()
setup_loader(a, dict(a='console.log("a")'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)
setup_loader(a)
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)

In [ ]:
#| export
def ensure_server(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=8000, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    **jupyuvi_kw, # Additional `JupyUvi` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]: # asdf
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, **jupyuvi_kw)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

async def ensure_server_a(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=8000, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]: # asdf
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, **jupyuvi_kw)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

In [ ]:
__mods__.clear()
port = free_port(6551, skip=pub_ports())
srv,app,rt,p = ensure_server(port=port, mods=dict(a='console.log("a")'))
srv2,app2,rt2,p2 = ensure_server(port=port, mods=dict(b='console.log("b")'))
cli = TestClient(app)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=9&debugger=true').text, 'debugger;\nconsole.log("a")')
test_eq(cli.get('/esm/b').text, 'console.log("b")')

await stop_server_a(port)

### Persistent server

```python
srv,app,rt,p = await ensure_server_a(port=None)

@rt
def hello(): return 'hello'

p('/hello') # or

display(P(hx_get='/hello', hx_trigger='load once'))

await stop_server_a(srv.port)
```

`ensure_server_a` returns persistent notebook state as `(srv, app, rt, p)`:

- `srv` controls the uvicorn lifecycle.
- `app` is the application actually being served.
- `rt` is its FastHTML route decorator.
- `p` previews routes through the returned server.

Selection precedence is:

1. An explicit live `srv`.
2. A live `JupyUvi` already bound to the requested port.
3. A newly created server.

When an explicit or discovered server is reused, `app`, `app_kw`, and `jupyuvi_kw` are ignored. For an explicit `srv`, its own port is authoritative and the requested `port` is ignored. `mods` are still registered, and the ESM route is installed on the reused app. `port=None` selects a free port outside Solveit’s public-port allocation.

The server remains live until explicitly stopped. In notebooks, prefer `await stop_server_a(srv.port)`, which supports synchronous and asynchronous servers and clears all namespace aliases to the stopped server.

That covers the main ESM loader behavior:

- versioned and unversioned routes
- caller-provided app route injection
- no duplicate route injection
- `ensure_server`/`ensure_server_a` passing `mods`
- reused server receiving additional `mods`
- missing module fallback
- `debugger=true` only applying when module source exists

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| exporti
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record):
        try: self.log(f"{(record.levelname+':'):<10}{self.format(record)}")
        except Exception: self.handleError(record)

In [ ]:
got = []
h = LoggerHandler(got.append)
h.emit(logging.LogRecord('x', logging.WARNING, '', 1, 'careful', (), None))
test_eq(got, ['WARNING:  careful'])

In [ ]:
class BadLog:
    def __call__(self, x): raise RuntimeError('boom')

h = LoggerHandler(BadLog())
h.handleError = lambda record: setattr(h, 'handled', True)
h.emit(logging.LogRecord('x', logging.INFO, '', 1, 'msg', (), None))
test_eq(h.handled, True)

In [ ]:
#| export
def setup_uvi_logging(
    _log:Callable[[str],object]|None=None, # Callable receiving formatted log lines; defaults to this module's `log`
    level:int=logging.INFO, # Level assigned to the uvicorn, uvicorn.error, and uvicorn.access loggers
) -> Callable[[],None]:
    "Route uvicorn logs to `_log`; return caller-owned cleanup and use `log_config=None` to preserve the setup."
    _log = log if _log is None else _log
    _lh = LoggerHandler(_log)
    names = ('uvicorn', 'uvicorn.error', 'uvicorn.access')
    prev = {}
    for name in names:
        lg = logging.getLogger(name)
        prev[name] = lg.handlers[:],lg.propagate,lg.level,lg.disabled
        lg.handlers,lg.propagate,lg.disabled = [_lh],False,False
        lg.setLevel(level)
    def cleanup():
        for name,(handlers,propagate,level,disabled) in prev.items():
            lg = logging.getLogger(name)
            lg.handlers,lg.propagate,lg.disabled = handlers,propagate,disabled
            lg.setLevel(level)
    return cleanup

Verify all three uvicorn loggers are changed and then restored.

In [ ]:
names = ('uvicorn','uvicorn.error','uvicorn.access')
logs = [logging.getLogger(o) for o in names]
old = [(o.handlers[:],o.propagate,o.level,o.disabled) for o in logs]
cleanup = setup_uvi_logging(lambda x: None, level=logging.WARNING)

test_eq([o.level for o in logs], [logging.WARNING]*3)
test_eq([o.propagate for o in logs], [False]*3)
test_eq([o.disabled for o in logs], [False]*3)
test_eq([len(o.handlers) for o in logs], [1]*3)

cleanup()
test_eq([(o.handlers,o.propagate,o.level,o.disabled) for o in logs], old)

In [ ]:
#| export
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port:int|None=8000, # Port to serve on; `None` chooses a free non-public port
    srv:JupyUvi|None=None, # Explicit live server; reused and never owned by this context
    reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
    after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
    before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback during cleanup, before owned-server shutdown
    restore_routes:bool=True, # Restore the initial route list only for a server created by this context
    timeout:float=5, # Seconds to wait for an owned server's port to become free
    mods:dict[str,str]|None=None, # Sources registered in the kernel-global ESM registry
    app_kw:dict|None=None, # Keywords used only when creating the default FastHTML app
    **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server
) -> AsyncIterator[JupyUvi]:
    "Async context manager yielding a live server; stop and optionally restore routes only when this context created it."
    srv,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, **jupyuvi_kw)
    routes = list(srv.app.routes) if own and restore_routes and hasattr(srv.app, 'routes') else None
    primary = None
    try:
        setup_loader(srv.app, mods)
        if after_start: await maybe_await(after_start(srv))
        yield srv
    except BaseException as e:
        primary = e
        raise
    finally:
        errs = []
        try:
            if own and before_stop: await maybe_await(before_stop(srv))
        except BaseException as e: errs.append(e)
        try:
            if routes is not None: srv.app.routes[:] = routes
        except BaseException as e: errs.append(e)
        try:
            if own:
                srv.server.should_exit = True
                deadline = (loop := asyncio.get_running_loop()).time() + timeout
                while not is_port_free(srv.port, srv.host):
                    if loop.time() > deadline: raise TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
                    await asyncio.sleep(0.1)
        except BaseException as e: errs.append(e)
        if errs:
            if primary is None:
                if len(errs) == 1: raise errs[0]
                raise BaseExceptionGroup("serve_app cleanup failures", errs)
            for e in errs: primary.add_note(f"serve_app cleanup failed: {type(e).__name__}: {e}")

In [ ]:
async with serve_app(port=None) as srv:
    test_eq(await is_serving_a(srv), True)
    test_eq(srv.port >= 6551, True)
    test_eq(srv.port in pub_ports(), False)
test_eq(is_port_free(srv.port), True)

Temporary live server and cleans it up.

In [ ]:
port = free_port(6551, skip=pub_ports())
base_srv,base_app,_,_ = await ensure_server_a(port=port)
called = []

async with serve_app(port=port, before_stop=called.append) as srv: test_is(srv, base_srv)

test_eq(called, [])
test_eq(await is_serving_a(base_srv), True)
await stop_server_a(port)

Existing servers are reused and not owned.

In [ ]:
port = free_port(6551, skip=pub_ports())

async with serve_app(port=port) as srv:
    n = len(srv.app.routes)
    @srv.app.get('/tmp-route')
    def tmp_route(): return 'ok'

test_eq(len(srv.app.routes), n)
test_eq(is_port_free(port), True)

Owned app routes are restored.

In [ ]:
class BadMods:
    def items(self): raise RuntimeError('setup boom')

port = free_port(6551, skip=pub_ports())
err = None
try:
    async with serve_app(port=port, mods=BadMods()): pass
except RuntimeError as e: err = e

test_eq(str(err), 'setup boom')
test_eq(is_port_free(port), True)

port = free_port(6551, skip=pub_ports())
err = None
async def bad_before(srv): raise RuntimeError('before boom')

try:
    async with serve_app(port=port, before_stop=bad_before) as srv:
        n = len(srv.app.routes)
        srv.app.get('/tmp-route')(lambda: 'ok')
        raise ValueError('body boom')
except ValueError as e: err = e

test_eq(str(err), 'body boom')
test_eq(err.__notes__, ['serve_app cleanup failed: RuntimeError: before boom'])
test_eq(len(srv.app.routes), n)
test_eq(is_port_free(port), True)

port = free_port(6551, skip=pub_ports())
started = asyncio.Event()
async def wait_forever():
    async with serve_app(port=port):
        started.set()
        await asyncio.Future()

task = asyncio.create_task(wait_forever())
await started.wait()
task.cancel()
err = None
try: await task
except asyncio.CancelledError as e: err = e

test_eq(type(err), asyncio.CancelledError)
test_eq(is_port_free(port), True)

INFO:     Waiting for application shutdown.


INFO:     Application shutdown complete.


INFO:     Finished server process [752]


Cleanup and route restoration still happen when teardown callback fails.

In [ ]:
# Add log_config=None to prevent Uvicorn calling logging.config.dictConfig() on startup
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

INFO:     Started server process [752]


INFO:     Waiting for application startup.


TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}


TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}


TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     Shutting down


The `log_config=None` kwarg passed in the test message prevents uvicorn from overwriting the custom `LoggerHandler` logging setup with its own `dictConfig` on startup.

In [ ]:
setup_uvi_logging()
await log.show(True)

[09:08:55.058] INFO:     Started server process [752]
[09:08:55.104] INFO:     Waiting for application startup.
[09:08:55.144] TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}
[09:08:55.179] TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}
[09:08:55.220] TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}
[09:08:55.256] INFO:     Application startup complete.
[09:08:55.290] INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
[09:08:55.433] INFO:     Shutting down


In [ ]:
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

In [ ]:
__mods__.clear()
async with serve_app(port=None, mods=dict(t='console.log("t")')) as srv:
    cli = TestClient(srv.app)
    test_eq(cli.get('/esm/t').text, 'console.log("t")')
    test_eq(sum(r.path == '/esm/{nm}' for r in srv.app.routes), 1)
test_eq(is_port_free(srv.port), True)

### Temporary server and testing

Use `serve_app` when a server should exist only for a block:

```python
async with serve_app(port=None) as srv:
    @srv.app.get('/hello')
    def hello(): return 'hello'

    cli = TestClient(srv.app)
    test_eq(cli.get('/hello').text, 'hello')
```

A server created by the context is owned by it: cleanup runs after normal exit, setup or body errors, and cancellation. By default, routes added to an owned app during the block are removed on exit.

A reused server is not owned by the context. It remains live afterward, and route changes made to it persist.

The context yields only the live server; use `srv.app`, `srv.port`, and other server attributes inside the block. In tests, `port=None` avoids persistent and publicly allocated ports.

`TestClient(srv.app)` tests the ASGI application directly and does not prove that uvicorn is reachable over the network. Use `await is_serving_a(srv)` when network availability is part of the contract.

## inject mod
Turn JavaScript stored in dialog messages into browser-loadable ES modules.

In [ ]:
#| export
async def get_js(
    id:str|None=None, # JavaScript message id; defaults to the preceding message
    dname:str='', # Optional dialog name containing the message
) -> str|None:
    "Strip the leading `%%js` or `%%javascript` line from a dialog message and return its remaining source."
    fl,_,src = (await read_msg(n=0 if id else -1, id=id, dname=dname)).content.partition('\n')
    if fl.lower() not in ('%%javascript', '%%js'): return None
    return src

In [ ]:
test_eq(await get_js(), 'a')

In [ ]:
#| exporti
def _py_assign(nm:str, src:str):
    "Return a Python assignment that round-trips `src`; reject invalid variable names."
    if not nm.isidentifier() or keyword.iskeyword(nm): raise ValueError(f"Invalid Python identifier: {nm!r}")
    return f'{nm} = {src!r}'

In [ ]:
src = r"""const single = \"'\"
const triple = "'''"
const path = String.raw`C:\tmp\file`
const tmpl = `line 1
line 2`
""" + "\\"
ns = {}
exec(_py_assign('_src', src), ns)
test_eq(ns['_src'], src)

for nm in ('not-valid', 'class', ''):
    with expect_fail(ValueError, 'Invalid Python identifier'): _py_assign(nm, src)

In [ ]:
#| export
@delegates(link_msg, but='content,msg_type')
async def add_js_msg(
    nm:str, # Valid Python identifier assigned to safely serialized source; invalid names raise `ValueError`
    id:str=None, # JavaScript source message; defaults to the preceding message and does not change the linkage anchor
    dname:str='', # Dialog to get info for; the linked code message is created in the current dialog
    **kwargs, # Delegated `link_msg` placement, execution, and display options; output is always a code message
) -> str:
    "Extract JavaScript from a dialog message and create or update a linked Python code message containing a safely serialized assignment."
    if (src := await get_js(id, dname)) is not None:
        return await link_msg(_py_assign(nm, src), msg_type='code', **kwargs)
    raise ValueError(f"Message #{id or (await read_msg()).id} does not contain Javascript source")

In [ ]:
await add_js_msg('_test_js', i_collapsed=True);

In [ ]:
_test_js = "const test = {'hello':'world'}\nglobalThis._test = test\nexport {test}"
#| linkedto: d954aa42

In [ ]:
#| export
@delegates(serve_app)
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int|None=None, # Numeric version selecting registered `<nm>_<v>` before falling back to `<nm>`,
    dname:str='', # Dialog owning the module; will be appended as query param if present
    **kwargs
) -> object:
    "Ensure the server, import `/esm/<nm>` in the active Solveit browser, then release any temporary server; browser and registry state persist."
    # url = f"{ifnone(url, f'/esm/{nm}')}{'?v='+str(v) if v is not None else ''}"
    q = filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else ''))
    url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
    async with serve_app(**kwargs) as _: return await js_eval_a(f"return await import({json.dumps(url)})")

In [ ]:
@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield None
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

In [ ]:
events = []

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a):
    test_eq(await load_mod('demo', 2), 'ok')

test_eq(events, [('enter', (), {}), ('eval', 'return await import("/esm/demo?v=2")'), ('exit',)])

In [ ]:
events = []

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a):
    test_eq(await load_mod('demo', dname='a b/c'), 'ok')

test_eq(events, [('enter', (), {}), ('eval', 'return await import("/esm/demo?dlg_name=a%20b%2Fc")'), ('exit',)])

In [ ]:
#| export
@delegates(serve_app)
async def setup_mod(
    nm:str, # Stable base ESM name
    src:str|None=None, # JavaScript source; defaults to the preceding JS message
    versioned:bool=False, # Use a kernel-local counter for `<nm>_<version>` and `?v=<version>` to bypass browser cache
    dname:str='', # Dialog owning the module; will be appended as query param if present
    **kwargs
) -> tuple[str,object]:
    "Register stable or versioned JavaScript, import it through ensured server, and return `(registered_name, browser_result)`."
    if src is None: src = await get_js()
    if src is None: raise ValueError(f"Message #{(await read_msg()).id} does not contain Javascript source")
    nmv = f"{nm}{'_'+str(v) if (v := str(kntr(nm)) if versioned else None) else ''}"
    reg_mod(nmv, src)
    return nmv, await load_mod(nm, v, dname=dname, **kwargs)

In [ ]:
loaded,res = await setup_mod('test', _test_js, True)
test_eq((await hx.eval_a('window._test.hello')), 'world')

empty,res = await setup_mod('empty', '')
test_eq(empty, 'empty')
test_eq(TestClient(mk_app({})).get('/esm/empty').text, '')
empty,res

('empty', {'data_id': '5ba06705-ae3a-4993-bdbc-f1d6e9c2dc5f', 'result': {}})

In [ ]:
reg_nm, res = await setup_mod('test', _test_js, True)
test_eq(reg_nm.split('_')[0], 'test')
test_eq(res.result, {'test': {'hello': 'world'}})
reg_nm, res

('test_2',
 {'data_id': 'b41d92df-16bd-4fac-94d2-f7bdeb7d71d7',
  'result': {'test': {'hello': 'world'}}})

In [ ]:
reg_nm, res = await setup_mod('test', _test_js)
test_eq(reg_nm, 'test')
reg_nm, res

('test',
 {'data_id': '82f5585a-25e5-4bda-b401-9d1f6b28f045',
  'result': {'test': {'hello': 'world'}}})

In [ ]:
await setup_mod('test2')

('test2',
 {'data_id': '05ca37d4-c83d-4fa9-85d9-73fcf2d2b4c1',
  'result': {'dname': 'prj/pote/nbs/02_server'}})

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,_,_ = await ensure_server_a(port=port, mods=dict(explicit='export const ok = true'))
test_eq(TestClient(app).get('/esm/explicit').text, 'export const ok = true')
await stop_server_a(port)

In [ ]:
with expect_fail(ValueError, 'does not contain Javascript source'):
    await setup_mod('test3')

The remaining local contract is that `setup_mod` registers the source, then passes `dname` through to `load_mod`. This test pins that down without invoking the browser:

In [ ]:
calls,res = [],[]

async def fake_load_mod(*args, **kwargs):
    calls.append((args, kwargs))
    return 'ok'

with mock_patch.dict(setup_mod.__globals__, load_mod=fake_load_mod):
    res.append(await setup_mod('named', 'export const x = 1', dname='a/b/c'))

test_eq(res, [('named', 'ok')])
test_eq(calls, [(('named', None), dict(dname='a/b/c'))])
test_eq(__mods__['named'], 'export const x = 1')

The examples establish the contract:

1. A module that exports `test` and assigns `window._test` is registered and imported; browser evaluation confirms `window._test.hello == 'world'`.
2. Repeated versioned loads create distinct module names (`test_1`, then `test_2`).
3. A plain `%%js` cell can be loaded from the immediately preceding message; its exported `dname` matches the current dialog path.
4. Calling `setup_mod` when the preceding message is not JavaScript raises a clear `ValueError`.

In short, it is a notebook/dialog development helper for: write JS in a message → register it on the local FastHTML server → dynamically import it into the current browser session.

### Browser ES modules

Register and import JavaScript in the active Solveit browser with:

```python
src = 'export const answer = 42'
nm,res = await setup_mod('editor', src, versioned=True)
```

With `versioned=True`, repeated calls register successive names such as `editor_1` and `editor_2`, and import distinct URLs such as `/esm/editor?v=1`. Use this during iterative development to bypass the browser’s ES-module cache.

The module registry belongs to the current Python kernel, not to an app or server. Registering an existing key replaces its source, and registered entries survive server shutdown. There is currently no public operation for unregistering an entry or clearing the registry.

Registered modules are executable browser code and may be served through the app’s public HTTP endpoint. Register only trusted source; module names are not an authorization boundary.

An unversioned name provides a stable URL. Replacing its registered source changes future HTTP responses, but a browser that has already imported that URL may continue using its cached module.

If given, `dname` is URL-encoded and forwarded as the `dlg_name` query parameter during the browser import. It does not affect module-registry lookup; browser-side code can use it only if it reads that parameter.

`setup_mod` and `load_mod` import through a given or discovered server in the active Solveit browser. If no live server is found and the requested port is free, a temporary server is created for the import and stopped afterward. If the port is occupied by a non-discoverable process, setup fails rather than replacing it.

JavaScript can also come from a dialog message whose complete first line is `%%js` or `%%javascript`:

- `get_js` returns the source without the magic line.
- `setup_mod` without `src` extracts and imports the preceding JavaScript message.
- `add_js_msg` creates linked Python state containing the extracted source.

A missing registry entry is served as a valid diagnostic JavaScript module rather than an HTTP error, so an import may succeed without providing the expected exports.

###### start-section

Let's test programatic import in the browser: `const {...} = await import('/esm/x?v=...')`

In [ ]:
await setup_mod('x', 'export const x=42,y="ok"', True)

('x_1',
 {'data_id': '5a64c1a0-0f7a-4844-86d6-ee44e4ec7b67',
  'result': {'x': 42, 'y': 'ok'}})

In [ ]:
res = await js_eval_a("""const {x,y} = await import('/esm/x?v=1'); return {x,y}""")
test_eq(res['result'], dict(x=42, y='ok'))

How could we use `import('x')` instead of `import('/esm/x?v=1')`

---

A bare specifier such as `import('x')` needs an **import map** in the document where the JavaScript runs. The existing `threejs_hdrs` is the same pattern.

For a fixed module version:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x?v=1'))), type='importmap'))
```

Then, after the browser has loaded that document:

```python
res = await js_eval_a("""const {x,y} = await import('x'); return {x,y}""")
test_eq(res['result'], dict(x=42, y='ok'))
```

For the unversioned, current module, map it to `/esm/x` instead:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x'))), type='importmap'))
```

Native import maps are fixed once module loading has begun. Thus `import('x')` is useful for a stable module identity, but versioned hot-reloads should keep using distinct URLs such as `/esm/x?v=1`, `/esm/x?v=2` (or use a shim that supports mutable import maps).

###### end-section

# pyskill

### Uvicorn logging

`setup_uvi_logging` routes the `uvicorn`, `uvicorn.error`, and `uvicorn.access` loggers to a callable and returns a cleanup function that restores their previous configuration.

Pass `log_config=None` when starting uvicorn; otherwise uvicorn may replace the custom logger configuration.

```python
cleanup = setup_uvi_logging(my_log)
try:
    async with serve_app(log_config=None) as srv:
        ...
finally:
    cleanup()
```

Logging configuration is process-global rather than server-owned, so the caller is responsible for cleanup.

In [ ]:
import pote.server

list_pyskills()['pote.server']

'Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.'

In [ ]:
doc(pote.server)

# module pote.server:

"""Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.

### Choosing a helper

```python
from pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg
```

- `await ensure_server_a(...)` starts or reuses a persistent FastHTML server and returns `(srv, app, rt, p)`.
- `async with serve_app(...)` is for a scoped server that is stopped automatically, particularly in tests.
- `find_server`, `is_serving_a`, and `stop_server_a` inspect and clean up existing server state.
- `free_port(public=True)` returns first available public port.
- `p(path)` previews a route in an iframe.
- `setup_mod(..., versioned=True)` supports iterative browser ES-module work.
- `get_js`, `add_js_msg`, and `setup_mod` without `src` extract JavaScript from a `%%js`/`%%javascript` message.

The async APIs are normally preferable in Solveit and Jupyter. These helpers target

In [ ]:
doc(pote.server.ensure_server)

def ensure_server(
    srv:fasthtml.jupyter.JupyUvi | None=None, # Explicit live server; its app and port win over all creation options
    port:int | None=8000, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:fasthtml.core.FastHTML | None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str, str] | None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict | None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    **jupyuvi_kw
)->tuple: # asdf"""Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."""

In [ ]:
doc(pote.server.serve_app)

def serve_app(
    app:fasthtml.core.FastHTML | None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port:int | None=8000, # Port to serve on; `None` chooses a free non-public port
    srv:fasthtml.jupyter.JupyUvi | None=None, # Explicit live server; reused and never owned by this context
    reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
    after_start:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
    before_stop:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # Sync or async callback during cleanup, before owned-server shutdown
    restore_routes:bool=True, # Restore the initial route list only for a server created by this context
    timeout:float=5, # Seconds to wait for an owned server's port to become free
    mods:dict[str, str] | None=None, # Sources registered in the kernel-global 

In [ ]:
doc(pote.server.setup_mod)

async def setup_mod(
    nm:str, # Stable base ESM name
    src:str | None=None, # JavaScript source; defaults to the preceding JS message
    versioned:bool=False, # Use a kernel-local counter for `<nm>_<version>` and `?v=<version>` to bypass browser cache
    dname:str='', # Dialog owning the module; will be appended as query param if present
    app:fasthtml.core.FastHTML | None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port:int | None=8000, # Port to serve on; `None` chooses a free non-public port
    srv:fasthtml.jupyter.JupyUvi | None=None, # Explicit live server; reused and never owned by this context
    reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
    after_start:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
    before_stop:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # S

# export -

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()

True